# Assignment

## Brief

Write the Python codes for the following questions.

## Instructions

Paste the answer as Python in the answer code section below each question.

### Question 1

Question: Implement a simple Thrift server and client that defines a `Student` struct with fields `name` (string), `age` (integer), and `courses` (list of strings). Include a service `School` with a method `enrollCourse` that takes a `Student` record and a course name, adds the course to the student's course list, and returns the updated `Student` record.

Answer:

**Thrift schema (student.thrift)**

In [1]:
%%writefile student.thrift

namespace py school

struct Student {
    1: required string name,
    2: required i32 age,
    3: required list<string> courses
}

service School {
    Student enrollCourse(1: Student student, 2: string courseName)
}

Writing student.thrift


**Thrift server (student_server.py)**

In [ ]:
# %%writefile: command to create a new file or owerwrite an existing file 
%%writefile student_thrift_server.py  

# the following codes will update the py file created from the writefile command with the content
import thriftpy2
from thriftpy2.rpc import make_server

student_thrift = thriftpy2.load("student.thrift", module_name="student_thrift")

class SchoolHandler:
    def enrollCourse(self, student, courseName):
        print(f"Server: Enrolling {student.name} in {courseName}")
        student.courses.append(courseName)
        return student

server = make_server(student_thrift.School, SchoolHandler(), '127.0.0.1', 9090)
print("Server listening on 127.0.0.1:9090...")
server.serve()

Writing student_thrift_server.py


Run the thrift server in terminal

**Thrift client (student_client.py)**

In [4]:
# Enter your student_client code here
import thriftpy2
from thriftpy2.rpc import make_client

# 1. Load the schema dynamically
student_thrift = thriftpy2.load("student.thrift", module_name="student_thrift")

# 2. Create the client
student_client = make_client(student_thrift.School, '127.0.0.1', 9090)

# 3. Create a Student object
alice = student_thrift.Student(
    name="Alice",
    age=20,
    courses=["Math", "History"]
)
print(f"Before enrollment: {alice.courses}")

# 4. Put the remote call inside a try block
try:
    # Call the remote method
    updated_alice = student_client.enrollCourse(alice, "Computer Science")
    
    # Print the result (only runs if no error occurs)
    print(f"After enrollment: {updated_alice.courses}")
    
except Exception as e:
    # This runs if an error occurs in the try block
    print(f"An error occurred: {type(e).__name__}: {e}")


Before enrollment: ['Math', 'History']
After enrollment: ['Math', 'History', 'Computer Science']


In [5]:
print("Server loaded these methods:", student_thrift.School.__dict__.keys())

Server loaded these methods: dict_keys(['__module__', '__dict__', '__weakref__', '__doc__', 'enrollCourse_args', 'enrollCourse_result', 'thrift_services'])


### Question 2

Question: Implement a simple Protocol Buffers server and client that defines a `Book` message with fields `title` (string), `author` (string), and `page_count` (integer). Include a service `Library` with a method `checkoutBook` that takes a `Book` message and returns the same `Book` message.

Answer:

**Protobuf schema (book.proto)**

In [7]:
%%writefile book.proto

syntax = "proto3";   // using version 3 of Protobuf

// Protobuf use "message" instead of "struct" to define data structures
message Book {
  string title = 1;
  string author = 2;
  int32 page_count = 3;
}

service Library {
  rpc checkoutBook(Book) returns (Book) {}
}

Overwriting book.proto


Run in terminal

### 📌📌📌 **Process Flow** 📌📌📌

1. You **write** book.proto (The schema).

2. You **run** protoc `python -m grpc_tools.protoc -I./ --python_out=. --grpc_python_out=. ./book.proto` in the terminal -> This auto-generates book_pb2.py (data) and book_pb2_grpc.py (network).
    - 2 book py files naming convention based on `--python_out` (flag creates _pb2.py) and `--grpc_python_out` (flag creates _pb2_grpc.py)
    - book_ based on schema name `book.proto`. If schema renamed to `librarybook.proto`, the 2 files generated will be `librarybook_pb2.py` and `librarybook_pb2_grpc.py`

3. You **write** `book_server.py` -> Inside this file, you `import` the two generated pb2 files. You use them to write your custom server logic.

4. You **run** `python book_server.py` in the terminal -> The server starts up, loads the pb2 templates, and listens for connections.

**Protobuf server (book_server.py)**

In [8]:
%%writefile book_server.py
from concurrent import futures
import grpc
import book_pb2
import book_pb2_grpc

# Implement the service logic
class LibraryServicer(book_pb2_grpc.LibraryServicer):
    def checkoutBook(self, request, context):
        print(f"Server: Checking out book '{request.title}' by {request.author}")
        # The prompt asks to return the same Book message
        return request

def serve():
    # Create a gRPC server
    server = grpc.server(futures.ThreadPoolExecutor(max_workers=10))
    # Add our servicer to the server
    book_pb2_grpc.add_LibraryServicer_to_server(LibraryServicer(), server)
    # Listen on port 50051
    server.add_insecure_port('[::]:50051')
    print("Server listening on port 50051...")
    server.start()
    server.wait_for_termination()

if __name__ == '__main__':
    serve()





Writing book_server.py


Run in terminal

**Protobuf client (book_client.py)**

In [9]:
# Enter your book_client code here
import grpc
import book_pb2
import book_pb2_grpc

# Connect to the server
with grpc.insecure_channel('localhost:50051') as channel:
    # Create a stub (client)
    stub = book_pb2_grpc.LibraryStub(channel)
    
    # Create a Book object
    my_book = book_pb2.Book(
        title="The Hitchhiker's Guide to the Galaxy",
        author="Douglas Adams",
        page_count=224
    )
    
    print(f"Client: Sending book '{my_book.title}' for checkout...")
    
    # Call the remote method
    response = stub.checkoutBook(my_book)
    
    # Print the result
    print("\nClient: Received response from server!")
    print(f"Title: {response.title}")
    print(f"Author: {response.author}")
    print(f"Page Count: {response.page_count}")

Client: Sending book 'The Hitchhiker's Guide to the Galaxy' for checkout...

Client: Received response from server!
Title: The Hitchhiker's Guide to the Galaxy
Author: Douglas Adams
Page Count: 224


### 📌📌📌 import 📌📌📌

**`import`** simply tells Python: *"Look in the folder right next to this script, find the file named `book_pb2.py`, read it, and make its code available for me to use."*

Here is exactly what happened:

1. **You generated them:** When you ran the `python -m grpc_tools.protoc ...` command in your terminal, the compiler read your `book.proto` file and **created two brand new Python files right there in your `assignment` folder**.

2. **They are local files:** If you look at your VS Code Explorer on the left, you should now see `book_pb2.py` and `book_pb2_grpc.py` sitting next to `book.proto` and `student_thrift_server.py`. 

3. **`import` opens them:** When you write `import book_pb2`, Python goes to your local folder, opens that file, runs the autogenerated code inside it, and gives you access to the `Book` class.

### Why does it look like a "standard template"?
The code inside those files is highly standardized, **auto-generated boilerplate**. It is not meant to be read or edited by humans. It's full of complex binary packing logic. 

However, it is **customized specifically for your `book.proto` file**. If you open `book_pb2.py`, you will see references to "title", "author", and "page_count". If you changed your `.proto` file and ran `protoc` again, it would rewrite those Python files with the new updates.
